# PhysIR — Stage 5: abl_percep @ 384 (DarkIR-loss-equivalent) · Colab Pro+

**The physics-vs-perceptual anchor.** Identical to the Stage 4 final model in EVERY way —
crop 384, batch 32, LR 5e-4, EMA, flip-only aug, 90/10 seed-42 split, 300 ep — EXCEPT the
physics losses are REPLACED by **LPIPS(VGG19) @ 0.01** = DarkIR's exact `L_percep` (Eq. 9).

This makes `full (Stage 4) vs abl_percep` the real **"PhysIR vs DarkIR-losses"** contrast at
the headline regime, and `abl_percep vs baseline` the perceptual-loss contribution.
(Base-loss caveat: our edge@0.05 vs paper λ_ed=50 — flagged in stage5_plan.md / guide G3.)

Extras vs the baseline run: **ep150 snapshot** (`snapshot_epochs`) for the dual
operating-point tables. Est. ~215–235 CU (LPIPS in the train step ≈ +20–30% over baseline's
181 CU). Drive-persistent + resume-safe: if CU runs dry, top up / wait for reset and re-run.


## 1. Runtime check

In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), 'No GPU — set Runtime → Change runtime type → GPU (A100/L4).'
print('CUDA:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

## 2. Mount Drive (persistent checkpoints + metrics)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Persistent output root — survives session disconnects. Everything we must keep goes here.
OUT_ROOT = '/content/drive/MyDrive/PhysIR/stage5_percep'
CKPT_DIR = os.path.join(OUT_ROOT, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_ROOT, '.metrics'), exist_ok=True)
print('Persistent output →', OUT_ROOT)

## 3. Install dependencies

In [ ]:
!pip install -q lpips pytorch-msssim einops ptflops pyiqa kaggle

## 4. Kaggle auth (Colab Secret) + pull code & datasets, then GUARD-CHECK

Uses the `COLAB_TOKEN` Colab Secret (🔑 left sidebar → Secrets) for Kaggle auth — no
kaggle.json upload. Add your 4 dataset slugs below.

In [ ]:
from google.colab import userdata
import os, sys, glob

# ── Kaggle auth via Colab Secret COLAB_TOKEN (your preferred method) ───────────
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
    f.write(userdata.get("COLAB_TOKEN"))
# also write kaggle.json if the secret is JSON creds (harmless fallback for the CLI)
try:
    _tok = userdata.get("COLAB_TOKEN")
    if _tok.lstrip().startswith("{"):
        with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
            f.write(_tok)
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
except Exception:
    pass

# ── EDIT: dataset slugs (owner/dataset) ───────────────────────────────────────
KAGGLE_CODE     = 'jeffreyboahen/physir-code'
KAGGLE_LOLBLUR  = 'jeffreyboahen/lolblur-dataset'
KAGGLE_LOLV2    = 'tanhyml/lol-v2-dataset'
KAGGLE_REALBLUR = 'jeffreyboahen/real-lolblur'      # EDIT to your Real-LOLBlur slug (or '')
KAGGLE_LSRW     = 'bjzh123/lsrw123'                 # LSRW generalization (Huawei+Nikon)
# ──────────────────────────────────────────────────────────────────────────────

# ABSOLUTE dest (we chdir to Drive later, so relative ./data would break)
DATA = '/content/data'
os.makedirs(DATA, exist_ok=True)

def pull(slug, sub):
    if not slug:
        print(f'  (skip {sub})'); return
    d = os.path.join(DATA, sub)
    if os.path.isdir(d) and os.listdir(d):
        print(f'  {sub}: already present'); return
    os.makedirs(d, exist_ok=True)
    print(f'  downloading {slug} -> {sub} ...')
    os.system(f'kaggle datasets download -d {slug} --unzip -p {d} -q')

pull(KAGGLE_CODE,     'code')
pull(KAGGLE_LOLBLUR,  'lolblur-dataset')
pull(KAGGLE_LOLV2,    'lolv2-dataset')
pull(KAGGLE_REALBLUR, 'real-lolblur')
pull(KAGGLE_LSRW,     'lsrw')

# ── locate roots (datasets often nest one level deep) ─────────────────────────
def find_root(base, marker, depth=3):
    if os.path.exists(os.path.join(base, marker)): return base
    for dd in range(1, depth+1):
        hits = sorted(glob.glob(os.path.join(base, *(['*']*dd), marker)))
        if hits: return hits[0][:-len(marker)].rstrip(os.sep)
    return None

CODE_ROOT = find_root(os.path.join(DATA, 'code'), 'train.py')
assert CODE_ROOT, 'train.py not found under the code dataset'
if CODE_ROOT not in sys.path: sys.path.insert(0, CODE_ROOT)

# ── GUARD: pulled code MUST contain the EMA + resume fixes ────────────────────
src = open(os.path.join(CODE_ROOT, 'train.py')).read()
missing = [n for n in ('class ModelEMA', 'restore_phys_state', "ckpt['ema']",
                       'grad_accum') if n not in src]
if missing:
    raise SystemExit(f'STALE physir-code - missing {missing}. '
                     f'Re-upload the physir-code Kaggle dataset with the current train.py.')
lsrc = open(os.path.join(CODE_ROOT, 'losses', 'loss.py')).read()
if 'class LPIPSLoss' not in lsrc:
    raise SystemExit('STALE physir-code - LPIPSLoss missing from losses/loss.py. Re-upload.')
print(f'\n[OK] CODE_ROOT = {CODE_ROOT}  (EMA + resume + LPIPSLoss present)')

LOLBLUR_ROOT = find_root(os.path.join(DATA, 'lolblur-dataset'),
                         os.path.join('train', 'high_sharp_scaled'))
assert LOLBLUR_ROOT, 'LOLBlur train not found'
print(f'LOLBLUR_ROOT = {LOLBLUR_ROOT}')
print('  has test/ :', os.path.isdir(os.path.join(LOLBLUR_ROOT, 'test')))

# strip macOS AppleDouble junk ('._*.png' crashes PIL — extension filters can't catch it)
!find /content/data \( -name '._*' -o -name '.DS_Store' \) -type f -delete 2>/dev/null


## 5. Train/val split (90/10, seed 42 — identical to all prior stages)

In [ ]:
import random
from pathlib import Path

SUBS      = ['low_blur_noise', 'high_sharp_scaled']
TRAIN_SRC = Path(LOLBLUR_ROOT) / 'train'
SPLIT     = Path('/content/LOLBlur_split')

random.seed(42)
clips = sorted([d.name for d in (TRAIN_SRC / 'low_blur_noise').iterdir() if d.is_dir()])
n_val = max(1, int(len(clips) * 0.1))
val_clips = set(random.sample(clips, n_val))
train_clips = [c for c in clips if c not in val_clips]
print(f'clips — train {len(train_clips)}, val {len(val_clips)}')

for name, cl in [('physir_train', train_clips), ('physir_val', val_clips)]:
    for sub in SUBS:
        base = SPLIT / name / sub; base.mkdir(parents=True, exist_ok=True)
        for c in cl:
            dst = base / c
            if not dst.exists():
                dst.symlink_to((TRAIN_SRC / sub / c).resolve())

PHYSIR_TRAIN = str(SPLIT / 'physir_train')
PHYSIR_VAL   = str(SPLIT / 'physir_val')
print('train:', PHYSIR_TRAIN, '\nval:  ', PHYSIR_VAL)

## 6. LOLv2 generalization dir (case-fix Test→test, same as Stage 2)

In [ ]:
LOLV2_SRC = os.path.join(DATA, 'lolv2-dataset')

def ci_child(parent, name):
    if not os.path.isdir(parent): return None
    for c in os.listdir(parent):
        if c.lower() == name.lower(): return os.path.join(parent, c)
    return None

LOLV2_ROOT = None
lolv2 = ci_child(LOLV2_SRC, 'LOL-v2') or ci_child(find_root(LOLV2_SRC, 'LOL-v2') or LOLV2_SRC, 'LOL-v2')
if lolv2:
    dst_root = '/content/LOLv2_gen'; linked = []
    for variant in ['Real_captured', 'Synthetic']:
        v = ci_child(lolv2, variant)
        t = ci_child(v, 'test') if v else None
        low, normal = (ci_child(t, 'Low'), ci_child(t, 'Normal')) if t else (None, None)
        if low and normal:
            dt = Path(dst_root) / 'LOL-v2' / variant / 'test'; dt.mkdir(parents=True, exist_ok=True)
            for sd, ss in [(dt/'Low', low), (dt/'Normal', normal)]:
                if not sd.exists(): sd.symlink_to(Path(ss).resolve(), target_is_directory=True)
            linked.append(variant)
    if linked:
        LOLV2_ROOT = dst_root
        print('LOLv2 gen ready:', linked, '→', dst_root)
if LOLV2_ROOT is None:
    print('LOLv2 not found — gen eval will be OFF (not fatal).')

## 7. Stage 4 config

In [ ]:
import yaml
RUN_NAME = 'stage5_percep'

cfg = {
    'datasets': {
        'name': 'LOLBlur',
        'train': {'low_path': f'{PHYSIR_TRAIN}/low_blur_noise',
                  'high_path': f'{PHYSIR_TRAIN}/high_sharp_scaled',
                  'crop_size': 384, 'batch_size': 16, 'n_workers': 8, 'use_flips': True},
        'val': {'test_path': PHYSIR_VAL, 'batch_size_test': 1, 'verbose': False, 'n_workers': 4},
    },
    'network': {'name': 'DarkIR', 'img_channels': 3, 'width': 32,
                'middle_blk_num_enc': 2, 'middle_blk_num_dec': 2,
                'enc_blk_nums': [1, 2, 3], 'dec_blk_nums': [3, 1, 1],
                'dilations': [1, 4, 9], 'extra_depth_wise': True},
    'save': {'pretrained': None,
             'path': f'{CKPT_DIR}/{RUN_NAME}_latest.pt',
             'best': f'{CKPT_DIR}/{RUN_NAME}_best.pt',
             'save_freq': 5, 'eval_freq': 10, 'gen_eval_freq': 20,
             'snapshot_epochs': [150],
             'perceptual_eval_epochs': [1, 50, 150, 300]},
    'pixel_criterion': 'l1',
    # DarkIR-loss-equivalent: LPIPS(VGG19) @ 0.01 = the paper's L_percep (Eq. 9)
    'perceptual': True, 'perceptual_weight': 0.01,
    'perceptual_type': 'lpips', 'perceptual_net': 'vgg',
    'perceptual_criterion': 'l1', 'perceptual_reduction': 'mean',
    'edge': True, 'edge_weight': 0.05, 'edge_criterion': 'l2', 'edge_reduction': 'mean',
    'frequency': False,
    'enhance': True, 'enhance_weight': 0.01, 'enhance_criterion': 'l1', 'enhance_reduction': 'mean',
    'physics_losses': {   # OFF — perceptual replaces physics (the G3 contrast)
        'enabled': False,
        'retinex': False, 'phase': False, 'blur': False,
    },
    # batch 16 x grad_accum 2 = effective batch 32 — LPIPS(VGG) activations at 384/b32
    # OOM'd the A100-80GB; accumulation reproduces batch-32 training exactly
    # (same mean gradient, same steps/epoch, same EMA update count), fp32, lr unchanged.
    'train': {'epochs': 300, 'lr_initial': 0.0005, 'weight_decay': 0.001,
              'grad_accum': 2,
              'lr_scheme': 'CosineAnnealing', 'eta_min': 1e-6, 'grad_clip': 1.0,
              'ema': {'enabled': True, 'decay': 0.9999, 'warmup': 2000}},
    'visual_samples': {'enabled': True,
                       'epochs': [1, 25, 50, 100, 150, 200, 300], 'num_samples': 6},
    'wandb': {'init': False, 'name': RUN_NAME},
    'Resize': False,
}
if LOLV2_ROOT:
    cfg['datasets_gen'] = {'name': 'All_LOL',
                           'train': {'n_workers': 2, 'verbose': False},
                           'val': {'test_path': LOLV2_ROOT, 'batch_size_test': 1,
                                   'verbose': False, 'n_workers': 2}}

cfg_path = f'{OUT_ROOT}/{RUN_NAME}.yml'
with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
pl = cfg['physics_losses']
print(f'config → {cfg_path}')
print(f'  crop 384 · batch 16 x accum 2 (eff 32) · lr 5e-4 · physics OFF + LPIPS(VGG19)@0.01 (abl_percep: DarkIR-loss-equivalent)')
print(f'  EMA {cfg["train"]["ema"]}')
print(f'  gen eval: {"ON" if LOLV2_ROOT else "OFF"}')

## 8. Resume detection (checkpoint lives on Drive → survives disconnects)

In [ ]:
latest = f'{CKPT_DIR}/{RUN_NAME}_latest.pt'
if os.path.isfile(latest):
    ck = torch.load(latest, map_location='cpu', weights_only=False)
    ep = ck.get('epoch', '?'); has_ema = 'ema' in ck
    sz = os.path.getsize(latest)/1e6
    print(f'✅ resuming from epoch {ep}  ({sz:.0f} MB, EMA in ckpt: {has_ema})')
    del ck
else:
    print('no checkpoint — training from scratch (epoch 1)')

## 9. Train

`.metrics` is written under Drive (cwd = OUT_ROOT). Single GPU → `python`, not torchrun.
**Watch epoch 1:** the memory fits check (no OOM) and the phase/base sanity — `loss_phase_freq /
loss_base` should be ≈ **0.72** (that's what phase=0.14 targets at crop 384). If OOM, set
`batch_size: 16` and `lr_initial: 0.00035` in cell 7 and re-run.\n\n> **Single GPU (Colab) → plain `python`, NOT torchrun.** train.py reads `WORLD_SIZE` (defaults to 1) so it takes the single-device path — no DDP / DistributedSampler / barriers. (The 2x-T4 torchrun path is Kaggle-only.)

In [ ]:
os.chdir(OUT_ROOT)   # → train.py writes .metrics/<run> under Drive
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # fragmentation headroom
print('cwd:', os.getcwd(), '\n')
!python {CODE_ROOT}/train.py -p {cfg_path}

## 9b. Epoch-1 phase/base sanity check (run after ≥1 epoch is logged)

In [ ]:
import pandas as pd
log = f'{OUT_ROOT}/.metrics/{RUN_NAME}/training_log.csv'
df = pd.read_csv(log)
r = df.iloc[0]
# Baseline has no phase loss (physics OFF), so no phase/base check — just confirm health.
print(f"ep1: total={r['loss_total']:.4f}  base={r['loss_base']:.4f}  "
      f"retinex={r['loss_retinex']:.4f}  phase={r['loss_phase_freq']:.4f}")
assert r['loss_total'] < 10 and r['loss_base'] > 0, 'ep1 sanity failed'
assert r['loss_phase_freq'] == 0 and r['loss_retinex'] == 0, 'physics should be OFF for baseline!'
print('[OK] baseline training healthy, physics confirmed OFF.')

## 10. Final evaluation (run once, at the end) — loads the **EMA** weights

Full-reference on the official LOLBlur **test** (1800) + LOLv2, and no-reference on
**Real-LOLBlur**. This is the only time the test set is touched.

In [ ]:
import torch, numpy as np
import train as T                      # module import runs defs only (guarded __main__)
from data import create_test_data

device = 'cuda'
best = f'{CKPT_DIR}/{RUN_NAME}_best.pt'
ck = torch.load(best, map_location='cpu', weights_only=False)
model = T.build_model(cfg['network']).to(device)
state = ck.get('ema') or ck['model']   # prefer EMA weights
print('loading', 'EMA' if 'ema' in ck else 'online', 'weights from best (epoch', ck.get('epoch'), ')')
model.load_state_dict({k.replace('module.', ''): v for k, v in state.items()}, strict=False)
model.eval()

lpips_fn = T._LPIPS(net='vgg', verbose=False).to(device) if T._LPIPS_AVAILABLE else None
results = {}

# ── full-reference: LOLBlur test ─────────────────────────────────────────────
LOLBLUR_TEST = os.path.join(LOLBLUR_ROOT, 'test')
opt_test = {'name': 'LOLBlur',
            'val': {'test_path': LOLBLUR_TEST, 'batch_size_test': 1},
            'train': {'n_workers': 4, 'verbose': True}}
test_loader, _ = create_test_data(rank=0, world_size=1, opt=opt_test)
tl = test_loader if not isinstance(test_loader, dict) else list(test_loader.values())[0]['loader']
results['LOLBlur_test'] = T.eval_metrics(model, tl, device, lpips_fn)
print('LOLBlur test :', results['LOLBlur_test'])

# ── full-reference: LOLv2 (if present) ───────────────────────────────────────
if LOLV2_ROOT:
    opt_v2 = {'name': 'All_LOL', 'val': {'test_path': LOLV2_ROOT, 'batch_size_test': 1},
              'train': {'n_workers': 2, 'verbose': False}}
    v2, _ = create_test_data(rank=0, world_size=1, opt=opt_v2)
    for key, ld in (v2.items() if isinstance(v2, dict) else [('lolv2', v2)]):
        loader = ld['loader'] if isinstance(ld, dict) else ld
        results[f'LOLv2_{key}'] = T.eval_metrics(model, loader, device, lpips_fn)
        print(f'LOLv2 [{key}]:', results[f'LOLv2_{key}'])

print('\n=== full-reference summary ===')
for k, v in results.items():
    print(f'  {k:20s} PSNR={v["psnr"]:.4f}  SSIM={v["ssim"]:.4f}  LPIPS={v["lpips"]:.4f}')

## 10b. No-reference: Real-LOLBlur (NIQE↓ / NRQM↑ / MUSIQ↑)

In [ ]:
# strip macOS AppleDouble junk ('._*.png' crashes PIL — extension filters can't catch it)
!find /content/data \( -name '._*' -o -name '.DS_Store' \) -type f -delete 2>/dev/null

REALBLUR_ROOT = find_root(os.path.join(DATA, 'real-lolblur'), 'realblur_dataset_test') \
                if os.path.isdir(os.path.join(DATA, 'real-lolblur')) else None
if REALBLUR_ROOT is None:
    print('Real-LOLBlur not found — skipping. (Set KAGGLE_REALBLUR / verify layout.)')
else:
    import pyiqa
    metrics = {'niqe': pyiqa.create_metric('niqe', device=device, as_loss=False),
               'nrqm': pyiqa.create_metric('nrqm', device=device, as_loss=False),
               'musiq': pyiqa.create_metric('musiq', device=device, as_loss=False)}
    opt_rb = {'name': 'RealBlur_Night',
              'val': {'test_path': REALBLUR_ROOT, 'batch_size_test': 1},
              'train': {'n_workers': 2, 'verbose': True}}
    rb, _ = create_test_data(rank=0, world_size=1, opt=opt_rb)
    rl = rb if not isinstance(rb, dict) else list(rb.values())[0]['loader']
    perc = T.eval_perceptual(model, rl, device, metrics)
    results['RealLOLBlur_noref'] = perc
    print(f"Real-LOLBlur:  NIQE={perc['niqe']:.4f}↓  NRQM={perc['nrqm']:.4f}↑  MUSIQ={perc['musiq']:.4f}↑")

## 10d. LSRW generalization (Huawei + Nikon, full-reference) — run AFTER cell 10

In [ ]:
# strip macOS AppleDouble junk ('._*.png' crashes PIL — extension filters can't catch it)
!find /content/data \( -name '._*' -o -name '.DS_Store' \) -type f -delete 2>/dev/null

import os, torch, numpy as np
import torchvision.transforms.functional as TF
from PIL import Image

LSRW_DIR = os.path.join(DATA, 'lsrw') if 'DATA' in globals() else '/content/data/lsrw'
os.makedirs(LSRW_DIR, exist_ok=True)
if not os.listdir(LSRW_DIR):
    os.system('kaggle datasets download -d bjzh123/lsrw123 --unzip -p %s -q' % LSRW_DIR)

IMG_EXT = {'.png', '.jpg', '.jpeg', '.bmp'}
def _sub(d, names):
    for c in os.listdir(d):
        if c.lower() in names and os.path.isdir(os.path.join(d, c)):
            return os.path.join(d, c)
    return None

# discover Eval Huawei/Nikon dirs with both low/ and high/, excluding 'Training data'
pairs = []
for root, dirs, _ in os.walk(LSRW_DIR):
    if 'training' in root.lower() or 'eval' not in root.lower():
        continue
    low, high = _sub(root, {'low'}), _sub(root, {'high', 'normal'})
    if low and high:
        pairs.append((os.path.basename(root), low, high))

def _pad8(x):
    _, _, h, w = x.shape
    ph, pw = (8 - h % 8) % 8, (8 - w % 8) % 8
    return torch.nn.functional.pad(x, (0, pw, 0, ph), mode='reflect'), h, w

@torch.no_grad()
def eval_pair(low_dir, high_dir):
    lo = sorted(f for f in os.listdir(low_dir)  if os.path.splitext(f)[1].lower() in IMG_EXT)
    hi = sorted(f for f in os.listdir(high_dir) if os.path.splitext(f)[1].lower() in IMG_EXT)
    assert len(lo) == len(hi) and lo, f'pair mismatch in {low_dir}'
    ps, ss, lp = [], [], []
    for lf, hf in zip(lo, hi):
        low  = TF.to_tensor(Image.open(os.path.join(low_dir,  lf)).convert('RGB')).unsqueeze(0).to(device)
        high = TF.to_tensor(Image.open(os.path.join(high_dir, hf)).convert('RGB')).unsqueeze(0).to(device)
        xin, H, W = _pad8(low)
        pred = model(xin, side_loss=False).clamp(0, 1)[:, :, :H, :W]
        Hm, Wm = min(H, high.shape[2]), min(W, high.shape[3])
        p2, h2 = pred[:, :, :Hm, :Wm], high[:, :, :Hm, :Wm]
        mse = torch.mean((p2 - h2) ** 2).item()
        ps.append(20 * np.log10(1.0 / (np.sqrt(mse) + 1e-8)))
        T._ssim_metric.to(device); ss.append(T._ssim_metric(p2, h2).item())
        lp.append(lpips_fn(p2 * 2 - 1, h2 * 2 - 1).mean().item())
    return {'psnr': float(np.mean(ps)), 'ssim': float(np.mean(ss)),
            'lpips': float(np.mean(lp)), 'n': len(ps)}

print('LSRW Eval pairs:', [(l, os.path.relpath(a, LSRW_DIR)) for l, a, _ in pairs] or 'NONE FOUND')
for label, low, high in pairs:
    r = eval_pair(low, high)
    results[f'LSRW_{label}'] = r
    print(f'LSRW {label:7s} ({r["n"]} imgs):  PSNR={r["psnr"]:.4f}  SSIM={r["ssim"]:.4f}  LPIPS={r["lpips"]:.4f}')

## 10e. Per-image metrics on LOLBlur test — enables the paired significance test (#12)

In [ ]:
import os, csv, torch, numpy as np

PERIMG_DIR = f'{OUT_ROOT}/per_image'
os.makedirs(PERIMG_DIR, exist_ok=True)
csv_path = f'{PERIMG_DIR}/per_image_{RUN_NAME}.csv'

if 'tl' not in globals():
    from data import create_test_data
    _opt = {'name': 'LOLBlur',
            'val': {'test_path': os.path.join(LOLBLUR_ROOT, 'test'), 'batch_size_test': 1},
            'train': {'n_workers': 4, 'verbose': False}}
    _t, _ = create_test_data(rank=0, world_size=1, opt=_opt)
    tl = _t if not isinstance(_t, dict) else list(_t.values())[0]['loader']

@torch.no_grad()
def per_image_eval(loader, out_csv):
    rows = []
    for i, (high, low) in enumerate(loader):
        high, low = high.to(device), low.to(device)
        pred = model(low, side_loss=False).clamp(0, 1)
        _, _, H, W = high.shape; pred = pred[:, :, :H, :W]
        mse = torch.mean((pred - high) ** 2).item()
        psnr = 20 * np.log10(1.0 / (np.sqrt(mse) + 1e-8))
        T._ssim_metric.to(device); ssim = T._ssim_metric(pred, high).item()
        lp = lpips_fn(pred * 2 - 1, high * 2 - 1).mean().item()
        rows.append((i, psnr, ssim, lp))
    with open(out_csv, 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['idx', 'psnr', 'ssim', 'lpips']); w.writerows(rows)
    return np.array([(p, s, l) for _, p, s, l in rows])

arr = per_image_eval(tl, csv_path)
print(f'per-image scores -> {csv_path}   ({len(arr)} images)')
print(f'means (must match cell-10 LOLBlur_test): '
      f'PSNR={arr[:,0].mean():.4f}  SSIM={arr[:,1].mean():.4f}  LPIPS={arr[:,2].mean():.4f}')

## 10c. Persist final results to Drive

In [ ]:
import json, os
out = {'run': RUN_NAME, 'best_epoch': int(ck.get('epoch', -1)),
       'weights': 'ema' if 'ema' in ck else 'online',
       'config_weights': {'phase': 0.14, 'retinex': 0.18, 'blur': False,
                          'crop': 384, 'batch': 32},
       'results': results,
       'per_image_csv': csv_path if ('csv_path' in globals() and os.path.exists(csv_path)) else None}
p = f'{OUT_ROOT}/stage4_final_results.json'
with open(p, 'w') as f:
    json.dump(out, f, indent=2, default=float)
print('saved ->', p)
if out['per_image_csv']:
    print('per-image CSV ->', out['per_image_csv'])
print(json.dumps(out['results'], indent=2, default=lambda x: round(float(x), 4)))

## 11. Figure generation — fixed test scenes (supplementary / ablation)

Individual full-res `[input | prediction | GT]` panels on **fixed test-scene indices** so every
ablation model renders identical scenes. Default: first 6 test items (edit `SCENE_IDS`).

In [ ]:
import torchvision.utils as vutils
SCENE_IDS = [0, 1, 2, 3, 4, 5]      # fixed across all ablation models
fig_dir = f'{OUT_ROOT}/figures/{RUN_NAME}'; os.makedirs(fig_dir, exist_ok=True)

saved = 0
with torch.no_grad():
    for i, (high, low) in enumerate(tl):
        if i > max(SCENE_IDS): break
        if i not in SCENE_IDS: continue
        high, low = high.to(device), low.to(device)
        pred = model(low, side_loss=False).clamp(0, 1)
        _, _, H, W = high.shape; pred = pred[:, :, :H, :W]
        panel = vutils.make_grid([low[0, :, :H, :W].cpu(), pred[0].cpu(), high[0].cpu()],
                                 nrow=3, padding=4)
        vutils.save_image(panel, f'{fig_dir}/scene_{i:03d}.png')
        vutils.save_image(pred[0].cpu(), f'{fig_dir}/scene_{i:03d}_pred_only.png')
        saved += 1
print(f'saved {saved} figure panels → {fig_dir}')
print('(re-run this cell for each ablation model with the SAME SCENE_IDS to build comparison figures)')